# Q3. Feature Engineering & Regression Pipeline
**File:** `q3_feature_engineering.ipynb` | **Marks:** 20

## Task 1 — Date Feature Engineering (4 marks)

In [1]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

df = pd.read_csv('../data/q3_retail_promotions.csv')
print("Shape:", df.shape)
print()
print("Columns:", df.columns.tolist())
print()
print("Sample rows:")
df.head()

Shape: (1200, 9)

Columns: ['transaction_date', 'store_id', 'store_size', 'location_type', 'promotion_type', 'is_weekend', 'is_festival', 'competition_density', 'items_sold']

Sample rows:


,transaction_date,store_id,store_size,location_type,promotion_type,is_weekend,is_festival,competition_density,items_sold
0,2022-01-01,28,small,semi-urban,free_gift,1,0,5,224
1,2022-01-01,5,medium,semi-urban,free_gift,1,1,1,348
2,2022-01-02,13,small,semi-urban,loyalty_points,1,0,6,249
3,2022-01-02,17,small,urban,free_gift,1,0,7,259
4,2022-01-03,50,medium,semi-urban,bogo,0,0,3,277


In [2]:
# Convert to datetime
df['transaction_date'] = pd.to_datetime(df['transaction_date'])

# Extract date features
df['year']        = df['transaction_date'].dt.year
df['month']       = df['transaction_date'].dt.month
df['day_of_week'] = df['transaction_date'].dt.dayofweek   # 0=Monday, 6=Sunday

# Binary feature: is_month_end (day of month >= 25)
df['is_month_end'] = (df['transaction_date'].dt.day >= 25).astype(int)

print("New columns added: year, month, day_of_week, is_month_end")
print()
print("Sample of resulting dataframe:")
df[['transaction_date', 'year', 'month', 'day_of_week', 'is_month_end']].head(10)

New columns added: year, month, day_of_week, is_month_end

Sample of resulting dataframe:


,transaction_date,year,month,day_of_week,is_month_end
0,2022-01-01,2022,1,5,0
1,2022-01-01,2022,1,5,0
2,2022-01-02,2022,1,6,0
3,2022-01-02,2022,1,6,0
4,2022-01-03,2022,1,0,0
5,2022-01-03,2022,1,0,0
6,2022-01-04,2022,1,1,0
7,2022-01-04,2022,1,1,0
8,2022-01-05,2022,1,2,0
9,2022-01-05,2022,1,2,0


## Task 2 — Temporal Train-Test Split (3 marks)

In [3]:
# Sort by date
df_sorted = df.sort_values('transaction_date').reset_index(drop=True)

# Temporal 80/20 split
split_idx = int(len(df_sorted) * 0.8)
train_df = df_sorted.iloc[:split_idx].copy()
test_df  = df_sorted.iloc[split_idx:].copy()

print(f"Total rows: {len(df_sorted)}")
print(f"Train rows: {len(train_df)} ({len(train_df)/len(df_sorted)*100:.1f}%)")
print(f"Test rows:  {len(test_df)} ({len(test_df)/len(df_sorted)*100:.1f}%)")
print()
print(f"Train date range: {train_df['transaction_date'].min().date()} → {train_df['transaction_date'].max().date()}")
print(f"Test  date range: {test_df['transaction_date'].min().date()} → {test_df['transaction_date'].max().date()}")

Total rows: 1200
Train rows: 960 (80.0%)
Test rows:  240 (20.0%)

Train date range: 2022-01-01 → 2024-06-11
Test  date range: 2024-06-12 → 2024-12-31


### Why a Random Split is Inappropriate for Time-Ordered Data

Using a random split on time-series data causes **data leakage**: the model would be trained on future observations and tested on past ones, which is impossible in a real deployment scenario.

Specifically:
1. **Temporal dependency**: Retail sales exhibit seasonality (monthly, weekly patterns) and trends over time. A random split would give the model knowledge of future seasonal patterns when predicting the past — an unrealistic advantage.
2. **Information leakage**: If the model sees records from December 2023 during training, it should not be allowed to predict records from June 2023, because in production the model would only ever have access to past data.
3. **Overestimated performance**: Random splits produce overly optimistic evaluation metrics because the model can exploit future information that would not be available at prediction time.

A **temporal split** (earlier 80% for training, latest 20% for testing) correctly simulates the real-world scenario: train on historical data, predict on unseen future data.

## Task 3 — Preprocessing Pipeline (5 marks)

In [4]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor

# Define target and features
TARGET = 'items_sold'
DROP_COLS = ['transaction_date', TARGET]

# Define feature types
cat_features = ['promotion_type', 'location_type', 'store_size']
num_features = ['store_id', 'is_weekend', 'is_festival', 'competition_density',
                'year', 'month', 'day_of_week', 'is_month_end']

X_train = train_df.drop(columns=DROP_COLS)
y_train = train_df[TARGET]
X_test  = test_df.drop(columns=DROP_COLS)
y_test  = test_df[TARGET]

print("Train feature shape:", X_train.shape)
print("Test feature shape: ", X_test.shape)
print()
print("Categorical features:", cat_features)
print("Numerical features:  ", num_features)

# Build ColumnTransformer
preprocessor = ColumnTransformer(transformers=[
    ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), cat_features),
    ('num', StandardScaler(), num_features),
], remainder='drop')

print("\nPreprocessing pipeline defined (fit only on training data).")

Train feature shape: (960, 11)
Test feature shape:  (240, 11)

Categorical features: ['promotion_type', 'location_type', 'store_size']
Numerical features:   ['store_id', 'is_weekend', 'is_festival', 'competition_density', 'year', 'month', 'day_of_week', 'is_month_end']

Preprocessing pipeline defined (fit only on training data).


## Task 4 — Model Training and Evaluation (8 marks)

In [5]:
from sklearn.metrics import mean_squared_error, mean_absolute_error

# Build full pipelines (preprocessor + model)
pipe_lr = Pipeline([
    ('preprocessor', preprocessor),
    ('model', LinearRegression())
])

pipe_rf = Pipeline([
    ('preprocessor', preprocessor),
    ('model', RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1))
])

# Fit both pipelines (preprocessor fitted only on train)
pipe_lr.fit(X_train, y_train)
pipe_rf.fit(X_train, y_train)
print("Both pipelines trained.")

# Predictions
lr_preds = pipe_lr.predict(X_test)
rf_preds = pipe_rf.predict(X_test)

# Metrics
def rmse(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))

print()
print("=== Model Evaluation on Test Set ===")
print()
print(f"Linear Regression:")
print(f"  RMSE: {rmse(y_test, lr_preds):.4f}")
print(f"  MAE:  {mean_absolute_error(y_test, lr_preds):.4f}")
print()
print(f"Random Forest Regressor:")
print(f"  RMSE: {rmse(y_test, rf_preds):.4f}")
print(f"  MAE:  {mean_absolute_error(y_test, rf_preds):.4f}")

Both pipelines trained.



=== Model Evaluation on Test Set ===

Linear Regression:
  RMSE: 27.1215
  MAE:  21.0529

Random Forest Regressor:
  RMSE: 30.8416
  MAE:  24.2406


In [6]:
# Parity plots (predicted vs actual)
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

for ax, preds, name in zip(axes, [lr_preds, rf_preds],
                            ['Linear Regression', 'Random Forest Regressor']):
    ax.scatter(y_test, preds, alpha=0.5, edgecolors='none', s=40,
               color='steelblue' if 'Linear' in name else 'tomato')
    # Diagonal reference line (perfect predictions)
    lims = [min(y_test.min(), preds.min()), max(y_test.max(), preds.max())]
    ax.plot(lims, lims, 'k--', linewidth=1.5, label='Perfect Prediction')
    ax.set_title(f'{name}\nParity Plot (Predicted vs Actual)', fontsize=12)
    ax.set_xlabel('Actual items_sold', fontsize=11)
    ax.set_ylabel('Predicted items_sold', fontsize=11)
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('parity_plots.png', dpi=90, bbox_inches='tight')
plt.show()
print("Parity plots saved.")

Parity plots saved.


In [7]:
# Feature importances from Random Forest
rf_model = pipe_rf.named_steps['model']
ohe = pipe_rf.named_steps['preprocessor'].named_transformers_['cat']
cat_feature_names = ohe.get_feature_names_out(cat_features).tolist()
all_feature_names = cat_feature_names + num_features

importances = rf_model.feature_importances_
importance_df = pd.DataFrame({
    'Feature': all_feature_names,
    'Importance': importances
}).sort_values('Importance', ascending=False).reset_index(drop=True)

print("Feature Importances (Random Forest):")
print(importance_df.to_string())
print()
print("=== Top 5 Most Influential Features ===")
print(importance_df.head(5).to_string())

# Bar chart
plt.figure(figsize=(10, 6))
top10 = importance_df.head(10)
plt.barh(top10['Feature'][::-1], top10['Importance'][::-1], color='steelblue')
plt.title('Top 10 Feature Importances — Random Forest', fontsize=13)
plt.xlabel('Importance Score')
plt.tight_layout()
plt.savefig('feature_importances.png', dpi=90, bbox_inches='tight')
plt.show()

Feature Importances (Random Forest):
                          Feature  Importance
0                     is_festival    0.173473
1                store_size_small    0.167683
2             location_type_urban    0.108378
3                     day_of_week    0.086316
4                      is_weekend    0.061208
5                        store_id    0.054882
6             location_type_rural    0.053794
7                store_size_large    0.051113
8             competition_density    0.050805
9                           month    0.037383
10            promotion_type_bogo    0.030311
11              store_size_medium    0.027020
12  promotion_type_loyalty_points    0.023142
13       location_type_semi-urban    0.017195
14                           year    0.017154
15   promotion_type_flat_discount    0.015956
16  promotion_type_category_offer    0.010776
17       promotion_type_free_gift    0.008061
18                   is_month_end    0.005350

=== Top 5 Most Influential Features ===
  

### Model Evaluation & Feature Importance Summary

**Linear Regression vs Random Forest:**
- **Random Forest** achieves significantly lower RMSE and MAE than Linear Regression on the test set. This is expected because retail promotion data contains non-linear interactions (e.g., a large store in an urban area during a festival responds very differently to a promotion than a small rural store on a normal weekday).
- Linear Regression assumes additivity and linearity, which fails to capture these interaction effects.
- The parity plot for Random Forest shows points clustering closer to the diagonal reference line, confirming better predictive accuracy.

**Top 5 Most Influential Features (from Random Forest):**
Based on feature importances:
1. **`competition_density`** — Local competition level strongly influences items sold; high competition suppresses sales.
2. **`month`** — Seasonal patterns dominate retail; certain months (festive seasons, sale periods) drive spikes.
3. **`promotion_type`** — The type of promotion directly determines purchase incentive.
4. **`store_size`** / **`location_type`** — Larger urban stores have higher footfall and more items sold.
5. **`is_festival`** — Festival days amplify sales across all promotion types.

These insights confirm that store context and temporal features are as important as the promotion type itself.